# CIR-ARC Phase 2: Upgraded Multi-Scale Neural Perception on Google Colab

This notebook trains and evaluates the **934,907-parameter Multi-Scale Object-Centric Perception Model** on Google Colab GPU (T4 / V100 / A100).

### Upgraded Architecture Components:
1. **ColorEmbedding (48-dim)**: Discrete ARC palette (0-9 + pad token 10) to continuous latent space
2. **MultiScaleCNNStem (286.5K params)**: 4-stage hierarchical depthwise-separable residual encoder + auxiliary boundary & cell-objectness heads
3. **Proposal-Guided SlotAttention (240K params)**: Top-K data-dependent proposal query initialization from objectness peaks
4. **Relational Set Transformer (265K params)**: 2-layer permutation-equivariant self-attention across 24 slot embeddings
5. **SlotMaskDecoder (25K params)**: Explicit spatial ownership mask decoding $(24 \times H \times W)$
6. **Symbolic Property Heads (38K params)**: Parallel MLPs predicting color, shape, size, position, orientation, symmetry
7. **ReconstructionDecoder (79K params)**: Mask-conditioned cross-attention reconstruction back to 2D grid

## 1. Hardware & Environment Check

In [ ]:
# Check assigned GPU
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Persistent Checkpointing)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/CIR_ARC_checkpoints/phase2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")

## 3. Clone Repository & Install Package

In [ ]:
# Clone or pull latest master branch from CIR-ARC repository
import os, sys
%cd /content
if not os.path.exists('/content/CIR-ARC'):
    !git clone https://github.com/Kapilraj-13/CIR-ARC.git
%cd /content/CIR-ARC
!git checkout master
!git pull origin master

# Ensure src is in python sys.path for the current kernel
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

# Install dependencies in editable mode
!pip install -e .
!pip install scipy matplotlib pyyaml tqdm pytest

## 4. Run Test Suite (Verify 330/330 Unit & Invariant Tests)

In [ ]:
!pytest -q

## 5. Verify Model Parameter Budget (934,907 Parameters)

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

from cir_arc.neural.training.trainer import PerceptionModel
model = PerceptionModel()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"PerceptionModel Total Trainable Parameters: {total_params:,}")
assert 800000 <= total_params <= 1200000, f"Parameter count {total_params} outside [800K, 1.2M]!"

## 6. Generate Synthetic Benchmark Data (12,000 Train / 3,000 Held-out)

In [ ]:
# Generate 12,000 training and 3,000 held-out validation tasks if not already generated
if not os.path.exists('data/synthetic/train') or len(os.listdir('data/synthetic/train')) < 1000:
    !python scripts/generate_data.py --n_per_rule 1000 --n_per_pair 500 --output_dir data/synthetic
else:
    print(f"Dataset already exists: {len(os.listdir('data/synthetic/train'))} train files, {len(os.listdir('data/synthetic/held_out'))} held-out files.")

## 7. Train Upgraded 935K Perception Model

In [ ]:
# Run Phase 2 perception training for 30 epochs with multi-objective losses
!python scripts/train_perception.py \
    --config configs/phase2.yaml \
    --epochs 30 \
    --batch_size 32 \
    --lr 0.001 \
    --device cuda \
    --checkpoint_dir /content/drive/MyDrive/CIR_ARC_checkpoints/phase2

## 8. Gate Evaluation & Ablation Comparison Table

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

import torch
from torch.utils.data import DataLoader
from cir_arc.neural.training.trainer import PerceptionModel
from cir_arc.neural.training.dataset import SyntheticArcDataset, collate_variable_grids
from cir_arc.neural.evaluation.perception_metrics import compute_perception_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PerceptionModel().to(device)

# Load best checkpoint from Google Drive
ckpt_path = '/content/drive/MyDrive/CIR_ARC_checkpoints/phase2/phase2_multiscale_slot_perception/best_model.pt'
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
    print("Loaded upgraded 935K model checkpoint!")

model.eval()
val_ds = SyntheticArcDataset(data_dir="data/synthetic/held_out")
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_variable_grids)

all_metrics = {}
with torch.no_grad():
    for batch in val_loader:
        grids = batch['input_grids'].to(device)
        masks = batch['input_masks'].to(device)
        gt_objects = batch['gt_objects']
        heights = batch['heights']
        widths = batch['widths']

        out = model(grids, mask=masks)
        batch_m = compute_perception_metrics(
            pred_logits=out['recon_logits'],
            target_grid=grids,
            objectness=out['objectness'],
            pred_props=out['props'],
            gt_objects_batch=gt_objects,
            mask=masks,
            heights=heights,
            widths=widths,
        )
        for k, v in batch_m.items():
            if isinstance(v, (int, float)):
                all_metrics.setdefault(k, []).append(v)

summary = {k: float(sum(v) / max(len(v), 1)) for k, v in all_metrics.items()}

print("=" * 60)
print("=== CIR-ARC PHASE 2 GATE EVALUATION RESULTS ===")
print("=" * 60)
print(f"Object F1 Score:       {summary.get('object_f1', 0.0):.3f} (Min Gate: >= 0.80, Strong: >= 0.85)")
print(f"Reconstruction Acc:    {summary.get('recon_acc', 0.0):.3f} (Min Gate: >= 0.85, Strong: >= 0.90)")
print(f"Color Accuracy:        {summary.get('color_acc', 0.0):.3f} (Min Gate: >= 0.95, Strong: >= 0.97)")
print(f"Position MAE:          {summary.get('pos_mae', 0.0):.3f} (Min Gate: <= 0.08, Strong: <= 0.05)")
print(f"Size MAE:              {summary.get('size_mae', 0.0):.3f}")
print("=" * 60)

## 9. Visual Inspection: Input vs Boundary vs Objectness vs Slot Masks vs Recon

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

ARC_COLORS = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]
cmap = mcolors.ListedColormap(ARC_COLORS)
norm = mcolors.Normalize(vmin=0, vmax=9)

# Sample 1 test example from held-out set
sample_batch = collate_variable_grids([val_ds[0]])
grids = sample_batch['input_grids'].to(device)
masks = sample_batch['input_masks'].to(device)
H = sample_batch['heights'][0]
W = sample_batch['widths'][0]

with torch.no_grad():
    out = model(grids, mask=masks)

input_np = grids[0, :H, :W].cpu().numpy()
recon_np = out['recon_logits'][0, :H, :W].argmax(dim=-1).cpu().numpy()
bound_np = out['boundary_map'][0, 0, :H, :W].cpu().numpy()
obj_np = out['cell_objectness'][0, 0, :H, :W].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(input_np, cmap=cmap, norm=norm)
axes[0].set_title("Ground Truth Input")
axes[1].imshow(bound_np, cmap='viridis')
axes[1].set_title("Predicted Boundary Map")
axes[2].imshow(obj_np, cmap='plasma')
axes[2].set_title("Cell Objectness Map")
axes[3].imshow(recon_np, cmap=cmap, norm=norm)
axes[3].set_title("Full Reconstruction")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Semantic Slot Retrieval & Object Memory Indexing (Phase 2 -> Phase 3 Bridge)

In [ ]:
import sys, os
sys.path.insert(0, '/content/CIR-ARC/src')
sys.path.insert(0, os.path.abspath('src'))

# Compute cosine similarities across slot embeddings to verify semantic clustering
import torch.nn.functional as F

with torch.no_grad():
    slots = out['slots'][0]  # (24, 128)
    norm_slots = F.normalize(slots, dim=-1)
    sim_matrix = torch.mm(norm_slots, norm_slots.t()).cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(sim_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title("Slot-to-Slot Relational Cosine Similarity")
plt.xlabel("Slot Index")
plt.ylabel("Slot Index")
plt.tight_layout()
plt.show()
print("Semantic Slot Cosine Diversity Index:", 1.0 - float(np.mean(np.abs(sim_matrix - np.eye(24)))))